In [6]:
# %pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb

In [7]:
from langchain_core.documents import Document

In [8]:
# PDF data

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("/Users/ravihw18/ML-Algorithms/14-RAG/Data/research.pdf")
# document=pdf_loader.load()
# document

## Ingestion Pipeline

In [9]:
# data -> document

import os
from langchain_community.document_loaders.pdf import PyMuPDFLoader

/var/folders/x8/szf89p9937d106fh4q2xp_8h0000gn/T/ipykernel_68166/2060057824.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.pdf import PyMuPDFLoader


### Documents

In [10]:
def load_all_pdfs():
    folder_path = "/Users/ravihw18/ML-Algorithms/14-RAG/Data"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyMuPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [11]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [12]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

### Chunks

In [13]:
# %pip install langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)

In [16]:
len(chunks)

321

### Embeddings

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [19]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


/var/folders/x8/szf89p9937d106fh4q2xp_8h0000gn/T/ipykernel_68166/4021223045.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("embedding dimensions=", self.model.get_sentence_embedding_dimension())


### Vector store

In [20]:
import chromadb
import uuid

In [21]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
            if len(documents) != len(embeddings):
                raise ValueError("num of documents does not match num of embeddings")
    
    
            # store => ids, embedding, document, metadata
            ids = []
            all_metadata = []
            documents_content = []
            embeddings_list = []
    
            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                doc_id = f"doc_{uuid.uuid4()}"
                ids.append(doc_id)
    
                metadata = dict(doc.metadata)
                metadata["doc_index"] = i
                metadata["content_length"] = len(doc.page_content)
                all_metadata.append(metadata)
    
                documents_content.append(doc.page_content)
    
                embeddings_list.append(embedding.tolist())
    
                self.collection.add(
                    ids=ids,
                    metadatas=all_metadata,
                    documents=documents_content,
                    embeddings=embeddings_list
                )
    
            print("total documents added in vector store=", len(documents_content))
            print("docs in collection:", self.collection.count())

In [22]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 642


In [23]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

embeddings shape: (321, 384)
total documents added in vector store= 321
docs in collection: 963


## Retrieval Pipeline

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store


    def retrieve(self, query, top_k=5, score_threshold=0.0):
        # query => embedding
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # semantic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        # cosine similarity
        retrieved_docs=[]
        
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank" : i + 1
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [26]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [27]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_556184c8-79c3-48f3-917c-21fb77d0843c',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7\nravihw18@gmail.com',
  'metadata': {'format': 'PDF 1.7',
   'author': 'Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin',
   'creator': '',
   'modDate': "D:20260908172617+00'00'",
   'page': 6,
   'keywords': '',
   'creationdate': '2026-09-08T17:26:17+00:00',
   'title': 'Attention is All you Need',
   'total_pages': 11,
   'trapped': '',
   'producer': 'pdfcpu v0.12.1 dev',
   'creationDate': "D:20260908172617+00'00'",
   'source': '/Users/ravihw18/ML-Algorithms/14-RAG/Data/research.pdf',
   'subject': 'Neural Information Processing Systems http://nips.cc/',
   'moddate': '2026-09-08T17:26:17+00:00',
   'file_path': '/Users/ravihw18/ML-Algorithms/14-RAG/Data/research.pdf',
   'doc_index': 50,
   'content_length': 131},
  'distance':

# Integration with LLMs

## Groq

In [ ]:
API_Key_GROQ = "PASTE-YOUR-API-KEY-HERE"

In [41]:
# %pip install langchain-groq

In [42]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_Key_GROQ,
    model="qwen/qwen3-32b",
    temperature=0.1,
    max_tokens=1024
)

In [43]:
# generate our retrieval-augmented output
def generate_output(query, retriever, llm, top_k=3):
    results = retriever.retrieve(query, top_k)

    context = "\n".join([doc["document"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context for the given query")

    # context + query
    prompt = f""" use given context to generate the answer for the query
                Context: {context}
                Query: {query} """

    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a list as prompt
    return response.content

In [ ]:
answer = generate_output("what is RAG?", rag_retriever, llm)

In [ ]:
print(answer)